# Finviz Stock Metrics Scraper

Select five different stock tickers in GitHub Actions. The notebook retrieves the same valuation metrics for each ticker using Finviz only.

**Fields:** Ticker, P/E, Forward P/E, P/S, P/C, P/B and P/FCF.

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
from pathlib import Path
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
from playwright.async_api import async_playwright
from IPython.display import display

TICKERS = [t.strip().upper() for t in os.getenv("TICKERS", "AMD,AAPL,MSFT,NVDA,GOOGL").split(",") if t.strip()]
if len(TICKERS) != 5:
    raise ValueError(f"Exactly five tickers are required; received {len(TICKERS)}.")
if len(set(TICKERS)) != 5:
    raise ValueError(f"Please choose five different tickers. Received: {TICKERS}")
if not all(re.fullmatch(r"[A-Z0-9.\-]{1,15}", t) for t in TICKERS):
    raise ValueError(f"One or more ticker symbols contain invalid characters: {TICKERS}")

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS = ["P/E", "Forward P/E", "P/S", "P/C", "P/B", "P/FCF"]
label_map = {"P/E": "p/e", "Forward P/E": "forward p/e", "P/S": "p/s", "P/C": "p/c", "P/B": "p/b", "P/FCF": "p/fcf"}
print("Selected stocks:", ", ".join(TICKERS))

## 1. Retrieve the Finviz snapshots

Each ticker is retrieved separately. If Finviz blocks access or changes its page structure, the notebook stops with a diagnostic message.

In [ ]:
async def scrape_finviz_snapshot(ticker: str) -> dict:
    url = f"https://finviz.com/quote.ashx?t={ticker}"
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(
            user_agent=("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) "
                        "Chrome/128.0.0.0 Safari/537.36"),
            viewport={"width": 1440, "height": 1000},
        )
        try:
            response = await page.goto(url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_timeout(2500)
            title = (await page.title()).lower()
            body = (await page.locator("body").inner_text()).strip()
            if response is not None and response.status >= 400:
                raise RuntimeError(f"Finviz returned HTTP {response.status} for {url}")
            if any(term in title for term in ["access denied", "attention required", "just a moment"]):
                raise RuntimeError(f"Finviz returned an access/challenge page for {ticker}.")
            if "snapshot" not in body.lower() and "forward p/e" not in body.lower():
                raise RuntimeError(f"Finviz snapshot was not detected for {ticker}; access may be blocked or the page layout may have changed.")

            pairs = {}
            tables = page.locator("table.snapshot-table2, table.js-snapshot-table, table")
            for table_index in range(await tables.count()):
                cells = await tables.nth(table_index).locator("td").all_inner_texts()
                cleaned = [re.sub(r"\s+", " ", value).strip() for value in cells]
                for i in range(0, len(cleaned) - 1, 2):
                    label, value = cleaned[i], cleaned[i + 1]
                    if label and value and label not in pairs:
                        pairs[label] = value
            if not pairs:
                raise RuntimeError(f"No label/value pairs extracted for {ticker}.")
            return pairs
        finally:
            await browser.close()

snapshots = {}
for ticker in TICKERS:
    print(f"Retrieving {ticker}...")
    snapshots[ticker] = await scrape_finviz_snapshot(ticker)
    print(f"Retrieved {len(snapshots[ticker])} fields for {ticker}.")

## 2. Select and save the requested metrics

In [ ]:
def to_number(value):
    text = str(value).strip().replace(",", "")
    if text in {"-", "", "N/A", "nan"}:
        return float("nan")
    if text.endswith("%"):
        text = text[:-1]
    multiplier = 1
    if text.endswith("B"):
        multiplier, text = 1e9, text[:-1]
    elif text.endswith("M"):
        multiplier, text = 1e6, text[:-1]
    elif text.endswith("K"):
        multiplier, text = 1e3, text[:-1]
    try:
        return float(text) * multiplier
    except ValueError:
        return float("nan")

rows = []
for ticker, snapshot in snapshots.items():
    normalized = {re.sub(r"\s+", " ", k).strip().lower(): v for k, v in snapshot.items()}
    missing = [label for label, key in label_map.items() if key not in normalized]
    if missing:
        raise ValueError(f"{ticker}: Finviz did not provide {missing}. Available labels: {sorted(normalized)}")
    row = {"Ticker": ticker, **{label: to_number(normalized[key]) for label, key in label_map.items()}}
    rows.append(row)
    pd.DataFrame([row]).to_json(OUTPUT_DIR / f"{ticker}_valuation_metrics.json", orient="records", indent=2)

metrics = pd.DataFrame(rows)
display(metrics)
print(f"Saved individual JSON files for {len(metrics)} stocks in {OUTPUT_DIR}.")

## 3. Visualize valuation multiples

One PNG is generated for each stock so that every company can be inspected independently.

In [ ]:
for _, row in metrics.iterrows():
    ticker = row["Ticker"]
    plot_data = row[METRICS].dropna()
    fig, ax = plt.subplots(figsize=(10, 5))
    if plot_data.empty:
        ax.text(0.5, 0.5, "No numeric valuation multiples available from Finviz",
                ha="center", va="center", transform=ax.transAxes)
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        plot_data.plot(kind="bar", ax=ax)
        ax.set_xlabel("Metric")
        ax.set_ylabel("Multiple (x)")
        ax.tick_params(axis="x", rotation=0)
    ax.set_title(f"{ticker} — Finviz valuation multiples")
    fig.tight_layout()
    chart_path = OUTPUT_DIR / f"{ticker}_valuation_multiples.png"
    fig.savefig(chart_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Chart saved to: {chart_path}")

## Notes on access and data quality

- Finviz remains the sole data provider.
- The workflow requires exactly five different tickers.
- Unavailable metric values are represented as missing values.
- HTTP errors, access challenges and unexpected page layouts raise explicit errors.
- The notebook does not attempt to bypass Finviz access controls.
- JSON and PNG files are saved individually for each selected stock.